# 211. Design Add and Search Words Data Structure
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/design-add-and-search-words-data-structure/

## 💡 Concepts

**Core concept(s):** A **trie** plus **DFS** to handle the `.` wildcard.

**Why it applies here:** `search` may contain `.`, which matches any single letter. In a trie, a normal letter follows one path; a `.` means "try every child here" — a small branching search (DFS) through the trie.

**Key intuition:** Follow letters straight down; at a dot, branch into all children and see if any path completes the word.

---

### 📚 What is a Trie (Prefix Tree)?
A **trie** stores words letter-by-letter along paths from a root, so words sharing a prefix share the same early path. A marker flags where a word ends.
- **Complexity:** insert / search a word of length L is **O(L)**, no matter how many words are stored.
- **In Python:** nested `dict`s (`{char: child}`) with a sentinel like `'$'` for word-ends.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- Tries.
- DFS/recursion to explore branches at a wildcard.

## 📝 Problem

Implement `addWord(word)` and `search(word)`, where `search` may include `.` matching any one letter.

**Example**
```
addWord("bad"); search("bad") -> True; search(".ad") -> True; search("b..") -> True; search("b.x") -> False
```

> Two approaches: a naive word-list scan and a trie with wildcard DFS.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """Level-order list -> tree (None = missing child), LeetCode style."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (in-order sorted, height ~log n)."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)
        node.right = helper(mid + 1, hi)
        return node
    return helper(1, n)

def preorder(root):
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    if not a and not b: return True
    if not a or not b or a.val != b.val: return False
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Word List + Match (naive)

**Idea:** Keep all words; on search, compare against every word of the same length, treating `.` as "matches anything".

**Time:** search `O(N·L)` (scans all words).

**Space:** `O(total letters)`.

In [ ]:
class WordDictNaive:
    """Naive: keep all words in a list and match each on search (slow for many words)."""
    def __init__(self):
        self.words = []
    def addWord(self, word: str) -> None:
        self.words.append(word)
    def search(self, word: str) -> bool:
        for w in self.words:               # compare against every stored word of equal length
            if len(w) == len(word) and all(a == "." or a == b for a, b in zip(word, w)):
                return True                # '.' matches any single character
        return False

### Approach 2 — Trie + Wildcard DFS (optimal for exact / typical)

**Idea:** Store words in a trie. For a plain letter, follow that child; for `.`, try all children with DFS. Exact searches are `O(L)`; dots add branching only where they appear.

**Time:** `O(L)` for dot-free searches (fast); more when dots branch.

**Space:** `O(total letters)`.

In [ ]:
class WordDictionary:
    """Trie-based: exact letters follow one path; a '.' branches into all children."""
    def __init__(self):
        self.root = {}                     # {letter: child_node}
    def addWord(self, word: str) -> None:
        node = self.root
        for c in word:
            node = node.setdefault(c, {})  # build the letter path
        node["$"] = True                   # mark the end of a word
    def search(self, word: str) -> bool:
        def dfs(node, i):
            if i == len(word):
                return "$" in node         # consumed the word -> is it a complete word here?
            c = word[i]
            if c == ".":                   # wildcard: any child could match this position
                return any(dfs(child, i + 1) for k, child in node.items() if k != "$")
            if c not in node:
                return False               # this exact letter isn't present
            return dfs(node[c], i + 1)     # follow the matching letter
        return dfs(self.root, 0)

In [ ]:
# Correctness check
for T in (WordDictNaive, WordDictionary):
    d = T()
    for w in ["bad", "dad", "mad"]:
        d.addWord(w)
    assert d.search("bad") is True
    assert d.search("pad") is False
    assert d.search(".ad") is True
    assert d.search("b..") is True
    assert d.search("b.x") is False
    print(T.__name__, "OK")
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on a growing number of words `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We insert the same words into both, then run one exact search per word.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def _words(n):
    out = []
    for i in range(n):
        s = ""; x = i
        for _ in range(5):
            s += chr(ord("a") + x % 26); x //= 26
        out.append(s)
    return out

def naive_bulk(words):
    d = WordDictNaive()
    for w in words: d.addWord(w)
    return sum(d.search(w) for w in words)     # N searches, each scans all -> O(N^2)

def trie_bulk(words):
    d = WordDictionary()
    for w in words: d.addWord(w)
    return sum(d.search(w) for w in words)     # N searches, each O(L) -> O(N)

def make_worst_case(n):
    return (_words(n),)

solutions = {
    "list (search O(N^2))": naive_bulk,
    "trie (search O(N)  )": trie_bulk,
}
sizes = [500, 1000, 2000, 4000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Trie + DFS for wildcards:** a dot means "branch to all children" — a tiny search inside the trie.
- **Signal:** "search with wildcards / partial match", "dictionary with `.`".
- **Related problems:** Implement Trie, Word Search II, Regular Expression Matching.
- **Common pitfalls:** (1) at a dot, iterating the end-marker key as if it were a child; (2) forgetting the length/end check when the word is consumed.